In [ ]:
import SimpleITK as sitk
import subprocess
import nibabel as nib
import os
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import convolve
np.set_printoptions(threshold=np.inf)

In [ ]:
MNI_Root=r"C:\Users\fhabi\Desktop\PhD\DataSet\Dataset\MNI152_T1_2mm.nii"
dataset_root=r"C:\Users\fhabi\Desktop\PhD\DataSet"
output_3D_nii_Dataset=r"C:\Users\fhabi\Desktop\PhD\DataSet\nii"
input_nii=r"C:\Users\fhabi\Desktop\PhD\DataSet\3D_nii_Dataset"
output_2D_nii_Dataset=r"C:\Users\fhabi\Desktop\PhD\DataSet\2D_nii_Dataset"
mri_root = r"C:\Users\fhabi\Desktop\PhD\DataSet\Dataset\MRI214\ADNI"
pet_root = r"C:\Users\fhabi\Desktop\PhD\DataSet\Dataset\PET214\ADNI"

In [ ]:
def load_nii_files_fromDataset(niiDatasetRoot):
    for subject in os.listdir(niiDatasetRoot):

        subject_path = os.path.join(niiDatasetRoot, subject)

        if not os.path.isdir(subject_path):
            continue

        for pair in os.listdir(subject_path):

            pair_path = os.path.join(subject_path, pair)

            mri_path = os.path.join(pair_path, "mri.nii.gz")
            pet_path = os.path.join(pair_path, "pet.nii.gz")

            # ✔ check files exist
            if not os.path.exists(mri_path):
                print("Missing MRI:", mri_path)
                continue

            if not os.path.exists(pet_path):
                print("Missing PET:", pet_path)
                continue

            # ✔ read images
            mri = sitk.ReadImage(mri_path)
            pet = sitk.ReadImage(pet_path)
            # To Do
            print("Loaded:", subject, pair)
#load_nii_files_fromDataset(input_nii)

In [ ]:
def scan_all_file_types(root_folder):
    extensions = {}

    for root, _, files in os.walk(root_folder):
        for f in files:
            ext = os.path.splitext(f)[1].lower()

            if ext == "":
                ext = "NO_EXTENSION"

            extensions[ext] = extensions.get(ext, 0) + 1

    print("File types:")
    for k, v in sorted(extensions.items(), key=lambda x: -x[1]):
        print(k, ":", v)

    return extensions
#scan_all_file_types(pet_root)

In [ ]:
def count_medical_folders(root_folder):
    dicom_folders = set()
    v_folders = set()
    i_folders = set()
    for root, _, files in os.walk(root_folder):

        has_dicom = False
        has_v = False
        has_i = False
        for f in files:
            f_lower = f.lower()

            # DICOM (file-based or extension-based)
            if f_lower.endswith(".dcm") or f_lower.endswith(".dicom"):
                has_dicom = True

            # V files
            if f_lower.endswith(".v"):
                has_v = True
            if f_lower.endswith(".i"):
                has_i = True

        # One time for each Folder
        if has_dicom:
            dicom_folders.add(root)

        if has_v:
            v_folders.add(root)

        if has_i:
            i_folders.add(root)

    print("Results:")
    print("DICOM folders:", len(dicom_folders))
    print("V folders:", len(v_folders))
    print("i folders:", len(i_folders))
    print("Total medical folders:", len(dicom_folders) + len(v_folders)+ len(i_folders))

    return dicom_folders, v_folders
#a=count_medical_folders(mri_root)
#b=count_medical_folders(pet_root)

In [ ]:
def print_unique_v_sizes(root_folder, dtype='>i2'):
    size_counts = {}   
    examples = {}     

    for root, _, files in os.walk(root_folder):
        for f in files:
            if f.lower().endswith(".i"):
                file_path = os.path.join(root, f)

                try:
                    file_size = os.path.getsize(file_path)

                    best_size = None

                    # Try dif Header
                    for header in [0, 1024, 2048, 4096, 8192]:
                        data = np.fromfile(file_path, dtype=dtype, offset=header)
                        size = data.size
                        print(size)

                        if best_size is None:
                            best_size = size

                    #Count
                    if best_size not in size_counts:
                        size_counts[best_size] = 0
                        examples[best_size] = file_path

                    size_counts[best_size] += 1

                except Exception as e:
                    print("❌ Error:", file_path, e)
    print(best_size)
    print("\nUnique sizes with counts:\n")

    for size, count in sorted(size_counts.items()):
        print(f"Size: {size}  →  Count: {count}")
        print(f"Example file: {examples[size]}")
        print("-" * 50)

    print("Total unique sizes:", len(size_counts))
#print_unique_v_sizes(pet_root)

In [ ]:
def load_Images_size(folder):

    reader = sitk.ImageSeriesReader()

    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    dicom_names = sorted(dicom_names)

    reader.SetFileNames(dicom_names)

    image = reader.Execute()

    # optional but recommended
    image = sitk.DICOMOrient(image, "LPS")

    # convert to numpy (z, y, x)
    arr = sitk.GetArrayFromImage(image)

    # number of slices
    z = arr.shape[0]
    y = arr.shape[1]
    x = arr.shape[2]

    print("Shape (z, y, x):", arr.shape)
    print("Number of slices (z):", z)

    return image, arr
#load_Images_size(pet_root)

In [ ]:
def type_conversion(image):
    image = sitk.Cast(image, sitk.sitkFloat32)
    return image

In [ ]:
def show_middle_slice(image, title=""):
    print("ITK size:", image.GetSize())

    arr = sitk.GetArrayFromImage(image)
    arr = np.transpose(arr, (1, 2, 0))
    mid_slice = arr[arr.shape[0] // 2]
    
    print(arr.shape)
    plt.imshow(mid_slice, cmap="gray")
    plt.title(title)
    plt.axis("off")
    plt.show()

In [ ]:
def Orientartion(image,type):
    orientation = sitk.DICOMOrientImageFilter_GetOrientationFromDirectionCosines(image.GetDirection())
    print("Detected "+type+" Image Orientartion:", orientation)                                                                

def Origin(image,type):
    origin= image.GetOrigin()
    print("Detected "+type+" Image Origin:", origin)

def Spacing(image,type):
    spacing=image.GetSpacing()
    print("Detected "+type+" Image Spacing:", spacing)

def Direction(image,type):
    direction=image.GetDirection()
    print("Detected "+type+" Image Direction:",direction)
    
def Image_Info(image,type):
    Orientartion(image,type)
    Origin(image,type)
    Spacing(image,type)
    Direction(image,type)
    
def Set_Image_Info(mri,pet):
    mri.SetOrigin(pet.GetOrigin())
    mri.SetSpacing(pet.GetSpacing())
    mri.SetDirection(pet.GetDirection())

In [ ]:
def register_pet_to_mri(pet, mri):
    
    # ---------- initial alignment ----------
    initial_transform = sitk.CenteredTransformInitializer(
        mri,   # fixed (target space)
        pet,   # moving
        sitk.Euler3DTransform(),                           #rotation چرخش and translation انتقال
        sitk.CenteredTransformInitializerFilter.GEOMETRY   #.GEOMETRY مرکز تصویر را بر اساس هندسه (ابعاد و سایز) تنظیم می‌کند  
        # به شدت پیکسل وابسته نیست (برخلاف MOMENTS) MOMENTS
    )
    #print(sitk.CenteredTransformInitializer(mri, pet, sitk.Euler3DTransform()))

    # ---------- registration method ----------
    registration = sitk.ImageRegistrationMethod()

    ## similarity metric (خوب برای multi-modality)
    registration.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)

    # optimizer      
    registration.SetOptimizerAsRegularStepGradientDescent(
        learningRate=0.5,
        minStep=1e-4,
        numberOfIterations=100,
        gradientMagnitudeTolerance=1e-6
    )
    registration.SetOptimizerScalesFromPhysicalShift()

    # interpolation
    registration.SetInterpolator(sitk.sitkLinear)

    # initial transform
    registration.SetInitialTransform(initial_transform, inPlace=False)

    # multiresolution     
    registration.SetShrinkFactorsPerLevel([4, 2, 1])
    registration.SetSmoothingSigmasPerLevel([2, 1, 0])
    registration.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    # ---------- run registration ----------
    final_transform = registration.Execute(mri, pet)
    # fixed=mri, moving=pet
    identity = sitk.Transform(3, sitk.sitkIdentity)
    # ---------- resample MRI into PET space ----------
    reference = sitk.Resample(
    mri,
    pet,
    sitk.Transform(),
    sitk.sitkLinear
)
 
    pet_registered = sitk.Resample(
    pet,
    mri,       
    final_transform,
    sitk.sitkLinear,
    0.0,
    pet.GetPixelID()
)
    return pet_registered

In [ ]:
def filter(image):
    image = sitk.Median(image, [3, 3, 3]) 
    image = sitk.DiscreteGaussian(image, variance=1.0)
    return image

In [ ]:
#--------------------------------------------------------------------

In [ ]:
def isFileExtension(extension,files):
    return any(f.lower().endswith(extension) for f in files)
def detect_format(folder):
    files = os.listdir(folder)
    
    if isFileExtension('.dcm',files):
        return ".dicom"
    elif isFileExtension('.v',files):
        return ".v"
    elif isFileExtension('.i',files):
        return ".i"
    elif isFileExtension('.hdr',files):
        return ".hdr"
    else:
        return None

In [ ]:
def is_dynamic_pet(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)

    reader.SetFileNames(dicom_names)

    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()

    reader.Execute()

    try:
        num_frames = int(reader.GetMetaData(0, "0054|0101"))  # Number of Time Slices
        return num_frames > 1
    except:
        return False

In [ ]:
def has_time_dimension(folder):
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)

    reader.MetaDataDictionaryArrayUpdateOn()
    reader.LoadPrivateTagsOn()
    reader.SetFileNames(dicom_names)

    reader.Execute()

    times = []

    for i in range(len(dicom_names)):
        if reader.HasMetaDataKey(i, "0054|1300"):  # Frame Reference Time
            times.append(reader.GetMetaData(i, "0054|1300"))

    return len(set(times)) > 1

In [ ]:
def detect_pet_type(folder):
    if is_dynamic_pet(folder):
        return "Dynamic PET"
    elif has_time_dimension(folder):
        return "Dynamic PET"
    else:
        return "Static PET"

In [ ]:
def load_dicom_file(folder,mode):
    frames=1
    reader = sitk.ImageSeriesReader()
    dicom_names = reader.GetGDCMSeriesFileNames(folder)
    n = len(dicom_names)
    if n == 0:
        return None

    if mode=="pet" and is_dynamic_pet(folder):
        dicom_names = dicom_names[:n // 6]
        frames=6
    reader.SetFileNames(dicom_names)
    try:
        image = reader.Execute()
    except:
        return None
    
    #image=crop_empty_space(image)
    image = sitk.DICOMOrient(image, "RAI")           #Orientation : RAS   #LPS
    
    image = sitk.Cast(image, sitk.sitkFloat32)       #Type conversion
    image = sitk.RescaleIntensity(image, 0, 1)

    #image = sitk.Median(image, [1, 1, 1])            #Filter denoising
    image = sitk.DiscreteGaussian(image, variance=1.0)
    original = image
    arr = sitk.GetArrayFromImage(image)              #Normalization
    arr = arr.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)
    image = sitk.GetImageFromArray(arr)
    image.CopyInformation(original)
    if mode=="pet" and (image.GetSize()==(128, 128, 47) or image.GetSize()==(400, 400, 109) or image.GetSize()==(128, 128, 31)):
        image = sitk.DICOMOrient(image, "LAS") 
        image.SetDirection([-1,0,0, 0,-1,0, 0,0,-1]) 
    return (image,frames)

In [ ]:
def load_v_file(folder,  mode="",frames=6, slices=63, rows=128, cols=128, header=0):
    v_files = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".v"):
                v_files.append(os.path.join(root, f))

    if len(v_files) == 0:
        raise FileNotFoundError("No .v file found")

    # read data
    file_path = v_files[0]
    data = np.fromfile(file_path, dtype='>i2', offset=header)
    data = data.astype(np.float32)
 
    #print("Raw Data size:", data.size)
    

    base = slices * rows * cols
    frames = data.size // base
    usable_size = frames * base
    #if (data.size-usable_size!=2048):
    #    print( data.size-usable_size)
    #    print("Raw Data size:", data.size)
    #    print("Detected frames:", frames)

    if frames == 0:
        raise ValueError("❌ Cannot reshape data")

    #trim extra data
    #data = data[-usable_size:]
    data = data[-usable_size:]
    #print("Original size:", data.size)
    #print("Usable size:", usable_size)
    #print("Removed extra:", data.size - usable_size)

    # reshape safely
    vol = data.reshape((frames, slices, rows, cols))

    # take first frame
    arr = vol[-1]
    arr = np.flip(arr, axis=1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)
    
    # convert to SimpleITK
    image = sitk.GetImageFromArray(arr)
    image = sitk.DICOMOrient(image, "RPS")
    if "30_min_3D_FDG_4i_16s" in v_files[0]:
        image.SetSpacing((2.1, 2.1, 2.4))
    else:
        image.SetSpacing((2.6, 2.6, 2.4))
    image.SetOrigin((128.0, 128.0, 75.6))
    image.SetDirection([-1,0,0, 0,-1,0,0,0,-1])
    return (image,frames)

In [ ]:
def load_i_file(folder, mode="",dtype=np.float32, slices=207, rows=256,cols=256,header=0):
    i_files = []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(".i"):
                i_files.append(os.path.join(root, f))

    if len(i_files) == 0:
        raise FileNotFoundError("No .i file found")
    
    # read data
    file_path = i_files[0]
    data = np.fromfile(file_path, dtype=dtype, offset=header)
    data = data.astype(np.float32)
    #print("Raw Data size:", data.size)

    base = slices * rows * cols

    frames = data.size // base
    usable_size = frames * base

    #print("Detected frames:", frames)

    if frames == 0:
        raise ValueError("❌ Cannot reshape data")

    data = data[-usable_size:]

    # reshape safely
    vol = data.reshape((frames, slices, rows, cols))

    # take first frame
    arr = vol[-1]
    arr = np.flip(arr, axis=1)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) 
    arr = np.where(arr < 0.01, 0, arr)  #all pix less than 0.01 set 0
    # convert to SimpleITK
    image = sitk.GetImageFromArray(arr)
    image = sitk.DICOMOrient(image, "RPI")
    image.SetSpacing((1.2, 1.2,1.2))
    image.SetOrigin((128.0, 128.0, 75.6))
    image.SetDirection([-1,0,0,
                  0,-1,0,
                  0,0,-1])
    return (image,frames)

In [ ]:
def Load_Images(folder, fmt, mode):

    if fmt == ".dicom":
        return load_dicom_file(folder, mode)
    elif fmt == ".v":
        return load_v_file(folder, mode)
    elif fmt == ".i":
        return load_i_file(folder, mode)
    else:
        raise ValueError("Unknown format")

In [ ]:
def convert_to_NumPy(img):
    if img is None:
        raise ValueError("Image is None")

    if isinstance(img, np.ndarray):
        return img

    return sitk.GetArrayFromImage(img)

In [ ]:
def show_three_views(vol,subject="",subjectId="",pair=""):
    vol = convert_to_NumPy(vol)
    z, y, x = vol.shape
    print(vol.shape)
    plt.figure(figsize=(12,4))

    plt.subplot(1,3,1)
    plt.imshow(vol[z//2], cmap='gray')
    plt.title("Axial")

    plt.subplot(1,3,2)
    plt.imshow(vol[:, y//2, :], cmap='gray')
    plt.title("Coronal")

    plt.subplot(1,3,3)
    plt.imshow(vol[:, :, x//2], cmap='gray')
    plt.title("Sagittal")

    plt.suptitle(f"Subject({subject}):{subjectId}, Pair({pair})")  

    plt.show()

In [ ]:
def get_slice(volume,sliceId=""):

    if sliceId=="":
         mid = volume.shape[0] // 2
         sl = volume[mid]
         sliceId=mid
    else:
         sl = volume[sliceId]

    return sl, sliceId   

In [ ]:
def get_slice_Id(volume,sliceId):
    if sliceId=="":
         mid = volume.shape[1] // 2
         sl = volume[:,:,mid]
         sliceId=mid
    else:
         sl = volume[:,:,sliceId]

    return sl

In [ ]:
def plot_pair_2D(mri, pet,title=""):

    # FORCE conversion ONLY ONCE
    mri=convert_to_NumPy(mri)
    pet=convert_to_NumPy(pet)

    print("MRI shape:", mri.shape)
    print("PET shape:", pet.shape)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(mri, cmap="gray" )
    plt.title(f"MRI")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(pet, cmap="gray" )
    plt.title(f"PET")
    plt.axis("off")

    plt.suptitle(title)
    plt.show()

In [ ]:
def plot_pair(mri, pet, sliceId,title=""):

    # FORCE conversion ONLY ONCE
    mri=convert_to_NumPy(mri)
    pet=convert_to_NumPy(pet)

    print("MRI shape:", mri.shape)
    print("PET shape:", pet.shape)

    mri_s, mri_idx = get_slice(mri, sliceId)
    pet_s, pet_idx = get_slice(pet, sliceId)
    
  

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.imshow(mri_s, cmap="gray" )
    plt.title(f"MRI (slice {mri_idx})")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(pet_s, cmap="gray" )
    plt.title(f"PET (slice {pet_idx})")
    plt.axis("off")

    plt.suptitle(title)
    plt.show()

In [ ]:
def Create_dataset_nii(mri_np,pet_np,subject,count):
    affine = np.eye(4)

    mri_nii = nib.Nifti1Image(mri_np, affine)
    pet_nii = nib.Nifti1Image(pet_np, affine)

    subject_out = os.path.join(output_dir, subject)
    pair_out = os.path.join(subject_out, f"pair_{count:02d}")

    mri_out_dir = os.path.join(pair_out, "mri")
    pet_out_dir = os.path.join(pair_out, "pet")

    os.makedirs(mri_out_dir, exist_ok=True)
    os.makedirs(pet_out_dir, exist_ok=True)

    mri_path = os.path.join(mri_out_dir, "mri.nii.gz")
    pet_path = os.path.join(pet_out_dir, "pet.nii.gz")

    nib.save(mri_nii, mri_path)
    nib.save(pet_nii, pet_path)

In [ ]:
def skull_strip_remove_neck(input_img, output_img, frac=0.3):
    cmd = [
        "bet",
        input_img,
        output_img,
        "-R",   # robust center estimation
        "-B",   # bias field + better cleanup
        "-f", str(frac),  # fractional intensity (try 0.25–0.4)
        "-g", "0",         # vertical gradient (important for neck removal)
        "-m"   # also output mask
    ]
    subprocess.run(cmd, check=True)

In [ ]:
def Load_png(path):

    img = plt.imread(path)
    return img

In [ ]:
def Load_MNI_Template_Image_File(MNI_Root):
     MNI_Template = sitk.ReadImage(MNI_Root)
     MNI_Template = sitk.DICOMOrient(MNI_Template, "LAI")
     MNI_Template = sitk.Cast(MNI_Template, sitk.sitkFloat32) 
     MNI_Template.SetDirection([-1,0,0, 0,-1,0, 0,0,-1])
     return MNI_Template

In [ ]:
def Create_3D_nii_Dataset(output_dir):

    MNI_Template=Load_MNI_Template_Image_File(MNI_Root)  

    os.makedirs(output_dir, exist_ok=True)
    totalPairImagesCount = 0

    subjects = os.listdir(mri_root)
    print("Total subjects:", len(subjects))

    for i, subject in enumerate(subjects):

        print("Subject",i+1,":", subject)

        eachSubjectPairImagesCount=1

        mri_path = os.path.join(mri_root, subject)
        pet_path = os.path.join(pet_root, subject)

        if not os.path.isdir(mri_path):
            continue
        if not os.path.exists(pet_path):
            continue

        if os.path.isdir(mri_path) and os.path.exists(pet_path):

            mri_folders = []
            pet_folders = []

            # collect all MRI folders
            for d, _, _ in os.walk(mri_path):
                fmt = detect_format(d)
                if fmt:
                    mri_folders.append(d)

            # collect all PET folders
            for d, _, _ in os.walk(pet_path):
                fmt = detect_format(d)
                if fmt:
                    pet_folders.append(d)

            # loop over pairs
            for idx,(mri_folder, pet_folder) in enumerate(zip(mri_folders, pet_folders)):
                
                mri_fmt = detect_format(mri_folder)
                pet_fmt = detect_format(pet_folder)
                mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
                pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")

                subject_out = os.path.join(output_dir, subject)
                os.makedirs(subject_out, exist_ok=True)

                pair_out = os.path.join(subject_out, f"pair_{idx+1}")
                os.makedirs(pair_out, exist_ok=True)

                # ✔
                mri= register_pet_to_mri(mri, MNI_Template)
                pet= register_pet_to_mri(pet, mri)

                # ✔ save correctly
                sitk.WriteImage(mri, os.path.join(pair_out, "mri.nii.gz"))
                sitk.WriteImage(pet, os.path.join(pair_out, "pet.nii.gz"))

                print(eachSubjectPairImagesCount)
                eachSubjectPairImagesCount+=1
            totalPairImagesCount += 1
    print ("Total Pair Images:",totalPairImagesCount) 
#Create_3D_nii_Dataset(output_3D_nii_Dataset)

In [ ]:
def Load_NIfTI_Image_File(path):
    img = sitk.ReadImage(path)
    if img.GetDimension() == 3:
        img = sitk.DICOMOrient(img, "RAI")
        img.SetDirection([-1,0,0, 0,-1,0, 0,0,-1])
    else:
        img.SetDirection([-1,0, 0,-1])
    img = sitk.Cast(img, sitk.sitkFloat32) 
    
    return img

In [ ]:
def Create_2D_nii_Dataset(input_nii,output_2D_nii_Dataset,sliceId):
    totalPairImagesCount=0
    for subjectCounter,subject in  enumerate(os.listdir(input_nii)):
        print("Subject",subjectCounter+1,":", subject)

        subject_path = os.path.join(input_nii, subject)

        if not os.path.isdir(subject_path):
            continue

        for pairCounter, pair in enumerate(os.listdir(subject_path)):

            pair_path = os.path.join(subject_path, pair)

            mri_path = os.path.join(pair_path, "mri.nii.gz")
            pet_path = os.path.join(pair_path, "pet.nii.gz")

            # ✔ check files exist
            if not os.path.exists(mri_path):
                print("Missing MRI:", mri_path)
                continue

            if not os.path.exists(pet_path):
                print("Missing PET:", pet_path)
                continue

            # ✔ read images
            mri = Load_NIfTI_Image_File(mri_path)
            pet = Load_NIfTI_Image_File(pet_path)

            subject_out = os.path.join(output_2D_nii_Dataset, subject)
            os.makedirs(subject_out, exist_ok=True)

            pair_out = os.path.join(subject_out, f"pair_{pairCounter+1}")
            os.makedirs(pair_out, exist_ok=True)
            mri_s = get_slice_Id(mri, sliceId)
            pet_s = get_slice_Id(pet, sliceId)

            #plot_pair_2D(mri_s, pet_s)

            # ✔ save correctly
            sitk.WriteImage(mri_s, os.path.join(pair_out, "mri.nii.gz"))
            sitk.WriteImage(pet_s, os.path.join(pair_out, "pet.nii.gz"))

            print(pairCounter+1)
            pairCounter+=1
        totalPairImagesCount += 1
    print("Total Pair Images:",totalPairImagesCount) 
#Create_2D_nii_Dataset(input_nii,output_2D_nii_Dataset,45)

In [ ]:
def load_nii_Dataset(input_nii):
    for subjectCounter,subject in  enumerate(os.listdir(input_nii)):

        subject_path = os.path.join(input_nii, subject)

        if not os.path.isdir(subject_path):
            continue

        for pairCounter, pair in enumerate(os.listdir(subject_path)):

            pair_path = os.path.join(subject_path, pair)

            mri_path = os.path.join(pair_path, "mri.nii.gz")
            pet_path = os.path.join(pair_path, "pet.nii.gz")

            # ✔ check files exist
            if not os.path.exists(mri_path):
                print("Missing MRI:", mri_path)
                continue

            if not os.path.exists(pet_path):
                print("Missing PET:", pet_path)
                continue

            # ✔ read images
            mri = Load_NIfTI_Image_File(mri_path)
            pet = Load_NIfTI_Image_File(pet_path)
            title=f"Subject({subjectCounter+1}):{subject}, Pair({pairCounter+1})"
            plot_pair_2D(mri, pet,title)
            #show_three_views(mri,subjectCounter+1,subject,pairCounter+1)
            #show_three_views(pet,subjectCounter+1,subject,pairCounter+1)
load_nii_Dataset(output_2D_nii_Dataset)

In [ ]:
def Hist(Image):

    plt.hist(Image, bins=100)

    plt.title("PET Histogram")
    plt.xlabel("Intensity")
    plt.ylabel("Voxel Count")
    plt.show()
    counts, bins = np.histogram(Image, bins=100)
    max_bin_idx = np.argmax(counts)
    left_idx = max(0, max_bin_idx - 4)
    right_idx = min(len(bins)-1, max_bin_idx + 5)
    bin_min = bins[left_idx]
    bin_max = bins[right_idx]
    mask = (Image >= bin_min) & (Image < bin_max)
    Image[mask] = 1
    arr = sitk.GetArrayFromImage(Image)
    arr = Image.copy()

  

    kernel = np.ones((2,2,2))

    count = convolve(arr.astype(np.int32), kernel, mode='constant')

    filtered = (count > 4).astype(np.uint8)
    return Image

In [ ]:
def plotThreeViewsMNITemplate(MNI_Root):
    MNI_Template=Load_MNI_Template_Image_File(MNI_Root)
    show_three_views(MNI_Template)
    print("Dimension:", MNI_Template.GetDimension())
    print("Size:", MNI_Template.GetSize())
    print("Spacing:", MNI_Template.GetSpacing())
    print("Origin:", MNI_Template.GetOrigin())
    print("Direction:", MNI_Template.GetDirection())
#plotThreeViewsMNITemplate(MNI_Root)

In [ ]:
MNI_Template=Load_MNI_Template_Image_File(MNI_Root)

totalPairImagesCount = 0
subjects = os.listdir(mri_root)
print("Total subjects:", len(subjects))

for i, subject in enumerate(subjects):
    #print("Subject",i+1,":", subject)
    eachSubjectPairImagesCount=1
    mri_path = os.path.join(mri_root, subject)
    pet_path = os.path.join(pet_root, subject)
   
    if not os.path.isdir(mri_path):
        continue
    if not os.path.exists(pet_path):
        continue

    if os.path.isdir(mri_path) and os.path.exists(pet_path):

        mri_folders = []
        pet_folders = []

        # collect all MRI folders
        for d, _, _ in os.walk(mri_path):
            fmt = detect_format(d)
            if fmt:
                mri_folders.append(d)

        # collect all PET folders
        for d, _, _ in os.walk(pet_path):
            fmt = detect_format(d)
            if fmt:
                pet_folders.append(d)

        #print(f"Found {len(mri_folders)} MRI and {len(pet_folders)} PET")

        # loop over pairs
        for mri_folder, pet_folder in zip(mri_folders, pet_folders):

            #print("Detected MRI Image folder:", mri_folder)
            #print("Detected PET Image folder:", pet_folder)
            
            mri_fmt = detect_format(mri_folder)
            #print("Detected MRI Image format:", mri_fmt)
            pet_fmt = detect_format(pet_folder)
            ##print("Detected PET Image format:", pet_fmt)
            #mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
            #pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
            #Image_Info(mri,"MRI")
            #if pet_fmt==".dicom" :
                #print("Subject",i+1,":", subject)
                #mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
                #pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
                #print(subject)
            #if(pet_frames!=6):
                #show_three_views(mri)
                #mri= register_mri_to_pet(mri, pet)
                #show_three_views(mri)
                #show_three_views(pet)
           # Image_Info(mri,"MRI1")
           # Image_Info(pet,"PET1")
           # mri, pet, mask=skull_strip_mri_pet(mri, pet)

            #print("Detected MRI Image Frames:", mri_frames)
            #print("Detected PET Image Frames:", pet_frames)

            #mri_np = convert_to_NumPy(mri)
            #pet_np = convert_to_NumPy(pet)
           
            #print("Detected MRI Image shape before crop:", mri_np.shape)
            #print("Detected PET Image shape before crop:", pet_np.shape)

            #print("MRI min:", mri_np.min())
            #print("MRI max:", mri_np.max())
            #print("PET min:", pet_np.min())
            #print("PET max:", pet_np.max())
            ##mri= remove_zero_padding(mri,0.3,"mri")
            #pet= remove_zero_padding(pet,0.2,"pet")
            
            #mri_crop_np = convert_to_NumPy(mri)
            #pet_crop_np = convert_to_NumPy(pet)

            #print("Detected MRI Image shape after crop:", mri_crop_np.shape)
            #print("Detected PET Image shape after crop:", pet_crop_np.shape)

            ##Create_dataset_nii(mri_np,pet_np,subject,pair_count)
            #print("MRI min:", mri_crop_np.min())
            #print("MRI max:", mri_crop_np.max())

            #print("PET min:", pet_crop_np.min())
            #print("PET max:", pet_crop_np.max())
            
            #show_three_views(mri)
            #show_three_views(pet)
            #print(mri_crop_np.dtype)
            #print(pet_crop_np.dtype)
            #plot_pair(mri, pet, title=subject)
            #mri = sitk.DICOMOrient(mri, "RAS")
            #pet = sitk.DICOMOrient(pet, "RAS")
   
           
            #x,y,z=pet_crop_np.shape
            #mri=downsample_image(mri,(z,y,x))
            #Set_Image_Info(mri,pet)
            #Image_Info(pet,"PET")
            #plot_pair(mri, pet, title=subject)
            #Set_Image_Info(mri,pet)
            #pet.SetDirection(mri.GetDirection())
            #pet.SetOrigin((0,0,0))
            #plot_pair(mri, pet, title=subject)
            pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
            #show_three_views(mri)
            if pet.GetSize()==(128, 128, 35) or pet.GetSize()==(128, 128, 31) or pet.GetSize()==(128, 128, 188) or pet.GetSize()==(400, 400, 109):
                #how_three_views(pet)
                #Image_Info(pet,"PET")
                mri, mri_frames = Load_Images(mri_folder, mri_fmt,"mri")
                #pet, pet_frames = Load_Images(pet_folder, pet_fmt,"pet")
                print("Subject",i+1,":", subject)
                #Image_Info(MNI_Template,"MNI_Template")
                mri= register_pet_to_mri(mri, MNI_Template)
                #show_three_views(mri)
                #Image_Info(mri,"mri")
                pet= register_pet_to_mri(pet, mri)
                #Image_Info(pet,"PETregi")
                #show_three_views(pet)
                plot_pair(mri, pet, title=f"Subject({i+1}) : {subject}, Pair({eachSubjectPairImagesCount})")
                eachSubjectPairImagesCount+=1
            #mri=force_spacing_111(mri)
            #show_three_views(pet)
            #print(mri_crop_np.dtype)
            #print(pet_crop_np.dtype)
            #plot_pair(mri, pet, title=subject)

            
            
            totalPairImagesCount += 1
            #print(count)
        #if count==2:
                #break

print ("Total Pair Images:",totalPairImagesCount)   